In [ ]:
%pip install segmentation-models-pytorch==0.5.0


In [ ]:
import json
import random
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset


In [ ]:
DATASET = Path("/kaggle/input/dataset-paneles")
SALIDA = Path("/kaggle/working/deeplabv3plus_resnet50")
CHECKPOINT = None

ARQUITECTURA = "DeepLabV3Plus"
ENCODER = "resnet50"
TAMANO = 512
BATCH_SIZE = 4
NUM_WORKERS = 2
USAR_AUMENTOS = False
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 300
PACIENCIA = 30
MAX_HORAS = 10.5
SEED = 42
UMBRAL = 0.5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MEDIA = np.array([0.485, 0.456, 0.406], dtype=np.float32)
DESVIACION = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def fijar_semilla(semilla: int = SEED) -> None:

    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(semilla)


def localizar_dataset(carpeta: Path) -> Path:

    requeridas = ["images/train", "images/val", "masks/train", "masks/val"]
    if all((carpeta / parte).is_dir() for parte in requeridas):
        return carpeta

    candidatas = sorted({
        imagenes.parent
        for imagenes in Path("/kaggle/input").rglob("images")
        if all((imagenes.parent / parte).is_dir() for parte in requeridas)
    })
    if len(candidatas) != 1:
        raise FileNotFoundError(
            f"Selecciona DATASET con images y masks. Candidatas encontradas: {candidatas}"
        )
    return candidatas[0]


In [ ]:
def aumentar_par(imagen: np.ndarray, mascara: np.ndarray) -> tuple:

    if random.random() < 0.5:
        imagen = np.fliplr(imagen)
        mascara = np.fliplr(mascara)

    giros = random.randrange(4)
    return (
        np.rot90(imagen, giros, axes=(0, 1)),
        np.rot90(mascara, giros, axes=(0, 1)),
    )


def preparar_imagen(imagen: np.ndarray) -> torch.Tensor:

    imagen_float = imagen.astype(np.float32) / 255.0
    imagen_norm = (imagen_float - MEDIA) / DESVIACION
    transpuesta = np.transpose(imagen_norm, (2, 0, 1))
    return torch.from_numpy(np.ascontiguousarray(transpuesta))


def preparar_mascara(mascara: np.ndarray) -> torch.Tensor:

    mascara_binaria = (mascara > 0).astype(np.float32)
    mascara_extendida = np.expand_dims(mascara_binaria, axis=0)
    return torch.from_numpy(np.ascontiguousarray(mascara_extendida))


In [ ]:
class DatasetPaneles(Dataset):

    def __init__(self, carpeta: Path, split: str, aumentar: bool = False):
        self.carpeta_imagenes = carpeta / "images" / split
        self.carpeta_mascaras = carpeta / "masks" / split
        self.imagenes = sorted(self.carpeta_imagenes.glob("*.png"))
        self.aumentar = aumentar

        if not self.imagenes:
            raise FileNotFoundError(f"No hay PNG en {self.carpeta_imagenes}")
        faltantes = [
            ruta.name for ruta in self.imagenes
            if not (self.carpeta_mascaras / ruta.name).is_file()
        ]
        if faltantes:
            raise FileNotFoundError(f"Faltan máscaras: {faltantes[:5]}")

    def __len__(self) -> int:
        return len(self.imagenes)

    def __getitem__(self, indice: int) -> tuple:
        ruta_imagen = self.imagenes[indice]
        ruta_mascara = self.carpeta_mascaras / ruta_imagen.name

        with Image.open(ruta_imagen) as archivo:
            imagen = archivo.convert("RGB")
        with Image.open(ruta_mascara) as archivo:
            mascara = archivo.convert("L")

        if imagen.size != (TAMANO, TAMANO):
            imagen = imagen.resize((TAMANO, TAMANO), Image.Resampling.BILINEAR)
        if mascara.size != (TAMANO, TAMANO):
            mascara = mascara.resize((TAMANO, TAMANO), Image.Resampling.NEAREST)

        imagen = np.array(imagen)
        mascara = np.array(mascara)
        if self.aumentar:
            imagen, mascara = aumentar_par(imagen, mascara)

        return preparar_imagen(imagen), preparar_mascara(mascara), ruta_imagen.name


def crear_cargadores(carpeta: Path) -> tuple:

    if BATCH_SIZE < 2:
        raise ValueError("Esta configuración de DeepLab requiere BATCH_SIZE de al menos 2")

    dataset_train = DatasetPaneles(carpeta, "train", aumentar=USAR_AUMENTOS)
    dataset_val = DatasetPaneles(carpeta, "val")
    if len(dataset_train) < BATCH_SIZE:
        raise ValueError("BATCH_SIZE supera el número de imágenes de entrenamiento")

    comunes = {
        "num_workers": NUM_WORKERS,
        "batch_size": BATCH_SIZE,
        "pin_memory": DEVICE.type == "cuda",
    }
    cargador_train = DataLoader(dataset_train, shuffle=True, drop_last=True, **comunes)
    cargador_val = DataLoader(dataset_val, shuffle=False, drop_last=False, **comunes)
    return cargador_train, cargador_val


In [ ]:
def crear_modelo(preentrenado: bool = True) -> nn.Module:

    constructor = getattr(smp, ARQUITECTURA)
    modelo = constructor(
        encoder_name=ENCODER,
        encoder_weights="imagenet" if preentrenado else None,
        in_channels=3,
        classes=1,
        activation=None,
    )
    return modelo.to(DEVICE)


class PerdidaSegmentacion(nn.Module):

    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode="binary", from_logits=True)

    def forward(self, logits: torch.Tensor, mascara: torch.Tensor) -> torch.Tensor:
        return self.bce(logits, mascara) + self.dice(logits, mascara)


def preparar_aprendizaje(modelo: nn.Module) -> tuple:

    criterio = PerdidaSegmentacion().to(DEVICE)
    optimizador = torch.optim.AdamW(
        modelo.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )
    planificador = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizador, mode="min", factor=0.5, patience=5, min_lr=1e-6,
    )
    escalador = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    return criterio, optimizador, planificador, escalador


In [ ]:
def entrenar_epoca(modelo, cargador, criterio, optimizador, escalador) -> float:

    modelo.train()
    suma_perdidas = 0.0
    n_imagenes = 0

    for numero, (imagenes, mascaras, _) in enumerate(cargador, 1):
        imagenes = imagenes.to(DEVICE, non_blocking=True)
        mascaras = mascaras.to(DEVICE, non_blocking=True)
        optimizador.zero_grad(set_to_none=True)

        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            logits = modelo(imagenes)
            perdida = criterio(logits, mascaras)

        if not torch.isfinite(perdida).item():
            raise FloatingPointError("Pérdida de entrenamiento no finita")

        escalador.scale(perdida).backward()
        escalador.step(optimizador)
        escalador.update()

        cantidad = imagenes.size(0)
        suma_perdidas += perdida.item() * cantidad
        n_imagenes += cantidad

        if numero % 200 == 0 or numero == len(cargador):
            print(f"  Train: lote {numero}/{len(cargador)}", flush=True)

    return suma_perdidas / n_imagenes


def validar_epoca(modelo, cargador, criterio) -> dict:

    modelo.eval()
    suma_perdidas = 0.0
    n_imagenes = 0
    tp = fp = fn = 0

    with torch.inference_mode():
        for imagenes, mascaras, _ in cargador:
            imagenes = imagenes.to(DEVICE, non_blocking=True)
            mascaras = mascaras.to(DEVICE, non_blocking=True)

            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                logits = modelo(imagenes)
                perdida = criterio(logits, mascaras)

            if not torch.isfinite(perdida).item():
                raise FloatingPointError("Pérdida de validación no finita")

            cantidad = imagenes.size(0)
            suma_perdidas += perdida.item() * cantidad
            n_imagenes += cantidad

            predichas = torch.sigmoid(logits) >= UMBRAL
            reales = mascaras >= 0.5
            tp += (predichas & reales).sum().item()
            fp += (predichas & ~reales).sum().item()
            fn += (~predichas & reales).sum().item()

    return {
        "perdida": suma_perdidas / n_imagenes,
        "dice": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 1.0,
        "iou": tp / (tp + fp + fn) if tp + fp + fn else 1.0,
        "precision": tp / (tp + fp) if tp + fp else 0.0,
        "recall": tp / (tp + fn) if tp + fn else 0.0,
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }


In [ ]:
def configuracion_experimento() -> dict:

    return {
        "arquitectura": ARQUITECTURA,
        "encoder": ENCODER,
        "tamano": TAMANO,
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "aumentos": USAR_AUMENTOS,
        "learning_rate_inicial": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "epocas_objetivo": EPOCHS,
        "paciencia": PACIENCIA,
        "max_horas_sesion": MAX_HORAS,
        "semilla": SEED,
        "umbral": UMBRAL,
        "media": MEDIA.tolist(),
        "desviacion": DESVIACION.tolist(),
        "dataset": str(DATASET),
        "torch": str(torch.__version__),
        "smp": smp.__version__,
        "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU",
        "perdida": "BCEWithLogitsLoss + DiceLoss",
    }


def estado_aleatorio() -> dict:

    numpy = np.random.get_state()
    return {
        "python": random.getstate(),
        "numpy": (numpy[0], numpy[1].tolist(), int(numpy[2]), int(numpy[3]), float(numpy[4])),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state() if DEVICE.type == "cuda" else None,
    }


def restaurar_aleatoriedad(estado: dict) -> None:

    random.setstate(estado["python"])
    numpy = estado["numpy"]
    np.random.set_state((
        numpy[0], np.asarray(numpy[1], dtype=np.uint32), numpy[2], numpy[3], numpy[4],
    ))
    torch.set_rng_state(estado["torch"].cpu())
    if DEVICE.type == "cuda" and estado["cuda"] is not None:
        torch.cuda.set_rng_state(estado["cuda"].cpu())


def guardar_checkpoint(estado: dict, destino: Path) -> None:

    destino.parent.mkdir(parents=True, exist_ok=True)
    torch.save(estado, destino)


def cargar_checkpoint(origen: Path) -> dict:

    return torch.load(origen, map_location=DEVICE, weights_only=True)


In [ ]:
def entrenar(checkpoint: Path | None = None) -> pd.DataFrame:

    if DEVICE.type != "cuda":
        raise RuntimeError("Activa una GPU en la configuración de Kaggle antes de entrenar")

    fijar_semilla()
    SALIDA.mkdir(parents=True, exist_ok=True)
    cargador_train, cargador_val = crear_cargadores(DATASET)
    modelo = crear_modelo(preentrenado=checkpoint is None)
    criterio, optimizador, planificador, escalador = preparar_aprendizaje(modelo)

    configuracion = configuracion_experimento()
    configuracion["n_train"] = len(cargador_train.dataset)
    configuracion["n_val"] = len(cargador_val.dataset)
    historial = []
    mejor_dice = -1.0
    sin_mejora = 0
    epoca_inicial = 1

    if checkpoint is not None:
        checkpoint = Path(checkpoint)
        estado = cargar_checkpoint(checkpoint)
        for clave in [
            "arquitectura", "encoder", "tamano", "aumentos", "media", "desviacion",
            "learning_rate_inicial", "weight_decay", "umbral", "semilla", "n_train", "n_val",
        ]:
            if estado["configuracion"][clave] != configuracion[clave]:
                raise ValueError(f"El checkpoint pertenece a otra configuración: {clave}")

        modelo.load_state_dict(estado["modelo"])
        optimizador.load_state_dict(estado["optimizador"])
        planificador.load_state_dict(estado["planificador"])
        escalador.load_state_dict(estado["escalador"])
        historial = estado["historial"]
        mejor_dice = estado["mejor_dice"]
        sin_mejora = estado["sin_mejora"]
        epoca_inicial = estado["epoca"] + 1

        anterior = checkpoint.with_name("best.pt")
        if not anterior.is_file():
            raise FileNotFoundError("Adjunta también best.pt junto al ultimo.pt anterior")
        if anterior.resolve() != (SALIDA / "best.pt").resolve():
            shutil.copy2(anterior, SALIDA / "best.pt")
        restaurar_aleatoriedad(estado["aleatoriedad"])
        configuracion["checkpoint_origen"] = str(checkpoint)
        print(f"Continuando desde la época {epoca_inicial}", flush=True)

    (SALIDA / "configuracion.json").write_text(
        json.dumps(configuracion, ensure_ascii=False, indent=2), encoding="utf-8",
    )
    print(f"GPU: {configuracion['gpu']}", flush=True)
    print(f"Train: {len(cargador_train.dataset)} · Val: {len(cargador_val.dataset)}", flush=True)

    inicio = time.monotonic()
    motivo = "epocas_objetivo"
    if sin_mejora >= PACIENCIA:
        motivo = "sin_mejora"
    else:
        for epoca in range(epoca_inicial, EPOCHS + 1):
            inicio_epoca = time.monotonic()
            perdida_train = entrenar_epoca(
                modelo, cargador_train, criterio, optimizador, escalador,
            )
            metricas = validar_epoca(modelo, cargador_val, criterio)
            planificador.step(metricas["perdida"])
            minutos = (time.monotonic() - inicio_epoca) / 60
            historial.append({
                "epoca": epoca,
                "perdida_train": perdida_train,
                **metricas,
                "learning_rate": optimizador.param_groups[0]["lr"],
                "minutos": minutos,
            })

            if metricas["dice"] > mejor_dice:
                mejor_dice = metricas["dice"]
                sin_mejora = 0
                torch.save(modelo.state_dict(), SALIDA / "best.pt")
            else:
                sin_mejora += 1

            estado = {
                "epoca": epoca,
                "modelo": modelo.state_dict(),
                "optimizador": optimizador.state_dict(),
                "planificador": planificador.state_dict(),
                "escalador": escalador.state_dict(),
                "mejor_dice": mejor_dice,
                "sin_mejora": sin_mejora,
                "historial": historial,
                "configuracion": configuracion,
                "aleatoriedad": estado_aleatorio(),
            }
            guardar_checkpoint(estado, SALIDA / "ultimo.pt")
            pd.DataFrame(historial).to_csv(SALIDA / "metricas.csv", index=False)
            print(
                f"Época {epoca}/{EPOCHS} · train {perdida_train:.4f} · "
                f"val {metricas['perdida']:.4f} · Dice {metricas['dice']:.4f} · "
                f"IoU {metricas['iou']:.4f} · {minutos:.1f} min",
                flush=True,
            )

            if sin_mejora >= PACIENCIA:
                motivo = "sin_mejora"
                break
            if (time.monotonic() - inicio) / 3600 >= MAX_HORAS:
                motivo = "limite_sesion"
                break

    resumen = {
        "motivo_fin": motivo,
        "epocas_terminadas": historial[-1]["epoca"] if historial else 0,
        "mejor_dice": mejor_dice,
        "horas_esta_sesion": (time.monotonic() - inicio) / 3600,
    }
    (SALIDA / "resumen.json").write_text(
        json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8",
    )
    print(f"Fin: {motivo} · Mejor Dice: {mejor_dice:.4f}", flush=True)
    return pd.DataFrame(historial)


In [ ]:
DATASET = localizar_dataset(DATASET)
print("Dataset:", DATASET)
print("Salida:", SALIDA)
print("Dispositivo:", DEVICE)

historial = entrenar(CHECKPOINT)


In [ ]:
def dibujar_historial(historial: pd.DataFrame) -> None:

    figura, ejes = plt.subplots(1, 2, figsize=(12, 4))
    ejes[0].plot(historial.epoca, historial.perdida_train, label="Train")
    ejes[0].plot(historial.epoca, historial.perdida, label="Validación")
    ejes[0].set_ylabel("BCE + Dice")
    ejes[1].plot(historial.epoca, historial.dice, label="Dice")
    ejes[1].plot(historial.epoca, historial.iou, label="IoU")
    ejes[1].set_ylabel("Solape")
    for eje in ejes:
        eje.set_xlabel("Época")
        eje.legend()
        eje.grid(alpha=0.2)
    figura.tight_layout()
    figura.savefig(SALIDA / "aprendizaje.png", dpi=150)
    plt.show()
    plt.close(figura)


def mostrar_predicciones(modelo, cargador, cantidad: int = 6) -> None:

    positivos = []
    negativos = []
    objetivo = cantidad // 2
    for imagenes, mascaras, nombres in cargador:
        for imagen, mascara, nombre in zip(imagenes, mascaras, nombres):
            destino = positivos if mascara.any().item() else negativos
            limite = objetivo if destino is positivos else cantidad - objetivo
            if len(destino) < limite:
                destino.append((imagen, mascara, nombre))
        if len(positivos) >= objetivo and len(negativos) >= cantidad - objetivo:
            break

    ejemplos = positivos + negativos
    figura, ejes = plt.subplots(len(ejemplos), 4, figsize=(14, 3 * len(ejemplos)), squeeze=False)
    modelo.eval()
    with torch.inference_mode():
        for fila, (imagen, mascara, nombre) in enumerate(ejemplos):
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                logits = modelo(imagen.unsqueeze(0).to(DEVICE))
            probabilidad = torch.sigmoid(logits)[0, 0].float().cpu().numpy()
            rgb = imagen.permute(1, 2, 0).numpy() * DESVIACION + MEDIA
            vistas = [np.clip(rgb, 0, 1), mascara[0].numpy(), probabilidad, probabilidad >= UMBRAL]
            titulos = [nombre, "Etiqueta", "Probabilidad", "Predicción"]
            for columna, (vista, titulo) in enumerate(zip(vistas, titulos)):
                if columna == 0:
                    ejes[fila, columna].imshow(vista)
                else:
                    ejes[fila, columna].imshow(vista, cmap="gray", vmin=0, vmax=1)
                ejes[fila, columna].set_title(titulo, fontsize=8)
                ejes[fila, columna].axis("off")
    figura.tight_layout()
    figura.savefig(SALIDA / "predicciones.png", dpi=150)
    plt.show()
    plt.close(figura)


def evaluar_mejor() -> None:

    modelo = crear_modelo(preentrenado=False)
    modelo.load_state_dict(torch.load(SALIDA / "best.pt", map_location=DEVICE, weights_only=True))
    cargador_val = DataLoader(
        DatasetPaneles(DATASET, "val"), batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
    )
    metricas = validar_epoca(modelo, cargador_val, PerdidaSegmentacion().to(DEVICE))
    (SALIDA / "metricas_mejor.json").write_text(
        json.dumps(metricas, ensure_ascii=False, indent=2), encoding="utf-8",
    )
    print("Mejor modelo:", metricas)
    mostrar_predicciones(modelo, cargador_val)


dibujar_historial(historial)
evaluar_mejor()
